# Exercise 4: storyboard → clips → measure consistency

## Tasks

1. **Plan:** let the OpenAI API write the prompt for one scene – Ueli with his dog at the lake.
2. **Generate:** create one clip in Google Flow with the character sheet as reference.
3. **Check:** compare the character in the clip with the sheet.

Required: 1 video. No Google Flow video available? Use Ueli's clip from Exercise 2.

**This is the solution.** Try `video_generation.ipynb` first.

## Setup

Select the **Python 3.11** kernel.

You need two things for this exercise:

- **OpenAI API key** – for the scene prompt (step 1), your own character sheet if you want one (step 2) and the consistency check (step 3). It comes from your `OPENAI_API_KEY` Codespaces secret.
- **Google account** – to sign in to Google Flow for the video (step 2). You create the clip by hand in the Google Flow web app, not in this notebook. One clip costs 10 of your 50 daily credits. No Google Flow clip? The notebook then uses Ueli's clip from Exercise 2.

This notebook uses the **OpenAI API** (model `gpt-6-luna`). Other providers, e.g. Google Gemini or Anthropic Claude, have their own APIs with different code. The API calls are in `helper.py` – open it to see how they work.

In [ ]:
# Standard Python libraries.
import sys
from pathlib import Path
from typing import Literal

# Libraries installed from requirements.txt.
import pandas as pd
from pydantic import BaseModel

# helper.py is in the main folder, one level up.
sys.path.append("..")

In [ ]:
# Functions from helper.py.
from helper import DATA, ask, create_pictures, extract_frames, show_images

## 1. Let the OpenAI API write the prompt for the scene

The scene: Ueli and his dog Bruno at the mountain lake. The model writes the five parts of a video prompt. The character descriptions are added **word for word** in front.

In [ ]:
# Fixed descriptions, added word for word to the prompt.
CHARACTER = "Ueli, red beanie, olive jacket"
DOG = "Bruno, Bernese, red collar"

In [ ]:
# The answer format: the five parts of a video prompt.
class ScenePrompt(BaseModel):
    shot: str
    camera: str
    action: str
    light: str
    sound: str

In [ ]:
# The instruction for the model.
PROMPT = f"""
Write one short video prompt for Google Flow.
The scene: {CHARACTER} together with the dog {DOG}
on the shore of a mountain lake.
Describe shot, camera, action, light and sound, each as one short sentence.
"""

In [ ]:
# Ask for the scene prompt. The empty list means: no images.
plan = ask(PROMPT, ScenePrompt, [])

# Print the five parts.
for part in plan:
    print(part, ":", plan[part])

In [ ]:
# The Google Flow prompt: the fixed descriptions first, then the five parts.
parts = [plan["shot"], plan["camera"], plan["action"], plan["light"], plan["sound"]]
flow_prompt = f"{CHARACTER}. {DOG}. " + " ".join(parts)
print(flow_prompt)

## 2. Create one clip in Google Flow with the character sheet

The character sheet is the reference picture: for Google Flow and for the check in step 3. Use the supplied sheet `ueli_sheet.jpg`, or set `OWN_SHEET = True` to let the OpenAI API create your own sheet from `CHARACTER` and `DOG` (model `gpt-image-2`, about US$0.006). It is saved as `reference/my_sheet.png`. If you change `CHARACTER`, also adapt the features in step 3.

In [ ]:
# The character sheet: the supplied one, or your own from the OpenAI API.
# Set this to True to create your own. One picture costs about US$0.006.
OWN_SHEET = False

if OWN_SHEET:
    sheet_prompt = (
        f"Character reference sheet of {CHARACTER}, together with the dog {DOG}. "
        "Front view and side view of both, full body, plain white background, "
        "photorealistic."
    )
    # Saved as reference/my_sheet.png. An unchanged prompt costs nothing again.
    sheet = create_pictures(["my_sheet"], [sheet_prompt], "reference")[0]
else:
    sheet = DATA / "reference" / "ueli_sheet.jpg"
show_images([sheet])

**Now the manual step in the Google Flow web app** ([labs.google/flow](https://labs.google/flow)). Use **Veo 3.1 Lite**, 8 seconds and **Ingredients to Video**. Add your character sheet as ingredient – `ueli_sheet.jpg` from `data/reference/` or your own `reference/my_sheet.png` – and paste the prompt from step 1. Upload the clip as `my_scene.mp4` next to this notebook.

In [ ]:
# Use your own clip. If there is none, use Ueli's clip from Exercise 2
# and the supplied sheet, which shows the same Ueli.
video_path = Path("my_scene.mp4")
if not video_path.exists():
    video_path = DATA / "videos" / "ueli_scene3_talk.mp4"
    sheet = DATA / "reference" / "ueli_sheet.jpg"
print("Using", video_path.name, "and", Path(sheet).name)

## 3. Compare the character in the clip with the sheet

One frame per second is compared with the character sheet.

In [ ]:
# Save one frame per second of the clip and show them.
paths = extract_frames(video_path)
show_images(paths)

In [ ]:
# The three allowed ratings.
Rating = Literal["match", "differs", "not visible"]


# The answer format: one rating per feature.
class Consistency(BaseModel):
    beanie: Rating
    jacket: Rating
    dog: Rating

In [ ]:
# The instruction: compare image 2 with the reference image 1.
PROMPT = """
Image 1 is the reference.
Rate the beanie, the jacket and the dog in image 2.
Use match for a visible match.
Use differs for a visible change.
Use not visible when it is unclear or out of frame.
"""

In [ ]:
# Compare every frame with the sheet. The sheet comes first.
rows = []
for path in paths:
    row = ask(PROMPT, Consistency, [sheet, path])
    row["frame"] = path.name
    rows.append(row)

# One row per frame.
ratings = pd.DataFrame(rows)
ratings

In [ ]:
# Count the frames that match the sheet, per feature.
for feature in ["beanie", "jacket", "dog"]:
    matches = (ratings[feature] == "match").sum()
    print(feature, ":", matches, "of", len(ratings), "frames match")

## Check your understanding

Answer briefly using the outputs above. Try first, then open the hints.

1. If every frame were generated as a separate image, the video would flicker (details jump from frame to frame). Why – and how do video models avoid it?
2. Why do we put the exact character description in front of the prompt, together with the character sheet?
3. Which feature matched least often in your clip? Look at these frames: is the character really different, or is the feature just not visible?
4. The ratings come from a model. How would you check whether you can trust them?
5. Why is the number of matching frames a better result than "the clip looks fine"?

**Your answers:**

1. …
2. …
3. …
4. …
5. …

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. Every image starts from new random noise, so details like the face or the clothes come out differently each time. Video models generate all frames together, across space and time. That keeps the details stable.

2. Small changes in the wording can produce a different person or dog. The same text plus the same reference image anchor the identity – also when you create more clips of Ueli later.

3. Use your table. `not visible` is not an error of the video: Ueli may be turned away or out of frame. Only `differs` shows that the character has changed.

4. Rate a few frames yourself and compare them with the ratings of the model. Where you and the model disagree, the model is not reliable.

5. A number can be compared between clips, prompts and settings, and other people can repeat the measurement. An impression cannot be compared.

</details>

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')